#Setup Env

In [1]:
# !pip install kaggle

In [2]:
!pip install icrawler

##Main Dataset

In [3]:
!kaggle datasets download -d rizkyyk/dataset-food-classification
!unzip dataset-food-classification.zip

Streaming output truncated to the last 5000 lines.
  inflating: dataset_gambar/train/French Fries/French Fries_069.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_070.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_071.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_072.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_075.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_076.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_077.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_078.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_080.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_081.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_083.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_086.jpg  
  inflating: dataset_gambar/train/French Fries/French Fries_087.jpg  
  inflating: dataset_gambar/train/Frenc

##Booster Dataset food101

In [4]:
!kaggle datasets download -d dansbecker/food-101
!unzip food-101.zip

Streaming output truncated to the last 5000 lines.
  inflating: food-101/food-101/images/tacos/1030289.jpg  
  inflating: food-101/food-101/images/tacos/1033196.jpg  
  inflating: food-101/food-101/images/tacos/1036030.jpg  
  inflating: food-101/food-101/images/tacos/1042175.jpg  
  inflating: food-101/food-101/images/tacos/1044043.jpg  
  inflating: food-101/food-101/images/tacos/1058697.jpg  
  inflating: food-101/food-101/images/tacos/1059239.jpg  
  inflating: food-101/food-101/images/tacos/1059326.jpg  
  inflating: food-101/food-101/images/tacos/1066762.jpg  
  inflating: food-101/food-101/images/tacos/1070967.jpg  
  inflating: food-101/food-101/images/tacos/1073468.jpg  
  inflating: food-101/food-101/images/tacos/1075296.jpg  
  inflating: food-101/food-101/images/tacos/1085243.jpg  
  inflating: food-101/food-101/images/tacos/108529.jpg  
  inflating: food-101/food-101/images/tacos/1086014.jpg  
  inflating: food-101/food-101/images/tacos/108945.jpg  
  inflating: food-101/f

##Booster Dataset Mendeley

In [5]:
!pip install gdown -q

In [6]:
!gdown https://drive.google.com/uc?id=15NakI-XT3pvJLMvUKyh3MtAMSLSTSVaV -O mendeley.zip

Downloading...
From (original): https://drive.google.com/uc?id=15NakI-XT3pvJLMvUKyh3MtAMSLSTSVaV
From (redirected): https://drive.google.com/uc?id=15NakI-XT3pvJLMvUKyh3MtAMSLSTSVaV&confirm=t&uuid=1a41f117-6ee5-4ff3-96b6-827164f23e36
To: /content/mendeley.zip
100% 110M/110M [00:01<00:00, 93.5MB/s]


In [7]:
!unzip -q mendeley.zip -d /content/mendeley_food

In [8]:
import os, shutil, random
import pandas as pd
from PIL import Image
from icrawler.builtin import BingImageCrawler

In [9]:
os.environ['KAGGLE_API_TOKEN'] = ""

#Setup Directory


In [10]:
indo_path = 'dataset_gambar/train'
food101_path = 'food-101/food-101/images'
final_path = 'data/vision/final'
extra_path = 'data/extra'
mendeley_path = '/content/mendeley_food/Indonesian Food Image/Original'

if os.path.exists(final_path):
    shutil.rmtree(final_path)

os.makedirs(final_path, exist_ok=True)
os.makedirs(extra_path, exist_ok=True)

print("Setup Done")

Setup Done


In [11]:
os.listdir(mendeley_path)

['gudeg',
 'gado_gado',
 'rendang',
 'bakso',
 'bebek_betutu',
 'nasi_goreng',
 'pempek',
 'sate',
 'rawon',
 'soto']

#Copy Data Indo

In [12]:
for cls in os.listdir(indo_path):
    if cls.startswith('.'):
        continue

    src = os.path.join(indo_path, cls)
    dst = os.path.join(final_path, cls)

    os.makedirs(dst, exist_ok=True)

    for f in os.listdir(src):
        if not f.startswith('.'):
            shutil.copy(os.path.join(src, f), dst)

print("Indo Copied")

Indo Copied


#Boost Data Indo (food101)

In [13]:
mapping_boost = {
    'Nasi Goreng': 'fried_rice',
    'Mie Goreng': 'ramen',
    'Pizza': 'pizza',
    'Burger': 'hamburger',
    'French Fries': 'french_fries',
    'Donat': 'donuts',
    'Pancake': 'pancakes'
}

MAX_PER_CLASS = 500

for indo_cls, food_cls in mapping_boost.items():
    src = os.path.join(food101_path, food_cls)
    dst = os.path.join(final_path, indo_cls)

    if not os.path.exists(src):
        continue

    os.makedirs(dst, exist_ok=True)

    existing = len(os.listdir(dst))
    needed = MAX_PER_CLASS - existing

    if needed <= 0:
        continue

    files = os.listdir(src)
    random.shuffle(files)

    for f in files[:needed]:
        shutil.copy(os.path.join(src, f), dst)

print("Boost done")

Boost done


#Scraping Additional Dataset

In [14]:
foods = {
    'Bakso': [
        "bakso kuah indonesia street food real photo",
        "indonesian bakso meatball soup",
        "bakso street food indonesia close up"
    ],
    'Ayam Geprek': [
        "ayam geprek crispy indonesia sambal",
        "indonesian fried chicken geprek real food",
        "ayam geprek street food indonesia"
    ],
    'Nasi Uduk': [
        "nasi uduk indonesia traditional food",
        "indonesian coconut rice nasi uduk",
        "nasi uduk street food jakarta"
    ],
    'Pecel': [
        "pecel sayur indonesia peanut sauce",
        "indonesian pecel salad traditional food"
    ],
    'Capcay': [
        "capcay indonesia stir fry vegetable",
        "indonesian capcay chinese food style"
    ],
    'Tempe Goreng': [
        "tempe goreng indonesia fried tempe",
        "indonesian tempe fried food close up"
    ]
}

for label, queries in foods.items():
    for q in queries:
        for i in range(3):
            crawler = BingImageCrawler(
                storage={'root_dir': f'data/extra/{label}'},
                downloader_threads=4
            )

            crawler.crawl(
                keyword=q + " real photo",
                max_num=120,
                min_size=(200,200)
            )

            print(f"{label} | {q} | run {i+1}")

print("Scraping done")

ERROR:downloader:Exception caught when downloading file https://www.alojapan.com/wp-content/uploads/2023/11/1701021731_maxresdefault.jpg, error: HTTPSConnectionPool(host='www.alojapan.com', port=443): Max retries exceeded with url: /wp-content/uploads/2023/11/1701021731_maxresdefault.jpg (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f771a78b620>: Failed to establish a new connection: [Errno 111] Connection refused')), remaining retry times: 2
ERROR:downloader:Exception caught when downloading file https://www.alojapan.com/wp-content/uploads/2023/11/1701021731_maxresdefault.jpg, error: HTTPSConnectionPool(host='www.alojapan.com', port=443): Max retries exceeded with url: /wp-content/uploads/2023/11/1701021731_maxresdefault.jpg (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f771a79ce30>: Failed to establish a new connection: [Errno 111] Connection refused')), remaining retry times: 1
ERROR:downloader:Exception caught when

Bakso | bakso kuah indonesia street food real photo | run 1
Bakso | bakso kuah indonesia street food real photo | run 2
Bakso | bakso kuah indonesia street food real photo | run 3
Bakso | indonesian bakso meatball soup | run 1
Bakso | indonesian bakso meatball soup | run 2
Bakso | indonesian bakso meatball soup | run 3
Bakso | bakso street food indonesia close up | run 1
Bakso | bakso street food indonesia close up | run 2
Bakso | bakso street food indonesia close up | run 3


ERROR:downloader:Response status code 403, file https://static.vecteezy.com/system/resources/previews/038/459/845/non_2x/ayam-geprek-sambal-bawang-crispy-fried-chicken-and-spicy-garlic-sauce-photo.jpg
ERROR:downloader:Response status code 404, file https://resepmamiku.com/en/wp-content/uploads/2023/05/ayam-geprek-indonesian-smashed-crispy-sambal-chicken-wondernonieskitchen-850x828.jpg


Ayam Geprek | ayam geprek crispy indonesia sambal | run 1
Ayam Geprek | ayam geprek crispy indonesia sambal | run 2
Ayam Geprek | ayam geprek crispy indonesia sambal | run 3
Ayam Geprek | indonesian fried chicken geprek real food | run 1
Ayam Geprek | indonesian fried chicken geprek real food | run 2
Ayam Geprek | indonesian fried chicken geprek real food | run 3
Ayam Geprek | ayam geprek street food indonesia | run 1
Ayam Geprek | ayam geprek street food indonesia | run 2
Ayam Geprek | ayam geprek street food indonesia | run 3


ERROR:downloader:Response status code 403, file https://static.vecteezy.com/system/resources/previews/026/552/948/non_2x/stock-of-nasi-uduk-food-foodgraphystudio-ai-generated-photo.jpg
ERROR:downloader:Response status code 403, file https://www.tasteatlas.com/images/dishes/036cef81f22f4aaba12208b08c1bda00.jpg
ERROR:downloader:Response status code 403, file https://cdn.tasteatlas.com/images/dishes/20af77e771d94c938cf7506ed8add7d7.jpg
ERROR:downloader:Response status code 403, file https://static.vecteezy.com/system/resources/previews/037/263/848/non_2x/nasi-uduk-betawi-steamed-rice-coconut-milk-seved-with-various-side-dish-photo.jpg
ERROR:downloader:Response status code 403, file https://static.vecteezy.com/system/resources/thumbnails/016/660/271/small_2x/nasi-uduk-or-steamed-rice-cooked-close-up-photo.jpg
ERROR:downloader:Response status code 403, file https://static.vecteezy.com/system/resources/previews/024/099/985/large_2x/nasi-uduk-is-an-style-steamed-rice-cooked-in-coconut-milk-us

...

Tempe Goreng | tempe goreng indonesia fried tempe | run 1


ERROR:downloader:Response status code 403, file https://www.shutterstock.com/shutterstock/photos/2144446937/display_1500/stock-photo-traditional-indonesian-culinary-food-tempe-goreng-or-fried-tempeh-served-on-ceramic-plate-with-2144446937.jpg


Tempe Goreng | tempe goreng indonesia fried tempe | run 2
Tempe Goreng | tempe goreng indonesia fried tempe | run 3


ERROR:downloader:Response status code 403, file https://static.vecteezy.com/system/resources/thumbnails/024/785/396/original/deep-fried-tempeh-or-tempe-goreng-is-a-traditional-food-from-indonesia-video.jpg


Tempe Goreng | indonesian tempe fried food close up | run 1
Tempe Goreng | indonesian tempe fried food close up | run 2
Tempe Goreng | indonesian tempe fried food close up | run 3
Scraping done


#Merge Data

In [15]:
TARGET_PER_CLASS = 400

def merge_dataset(source_path):
    for cls in os.listdir(source_path):
        src = os.path.join(source_path, cls)
        dst = os.path.join(final_path, cls)

        if not os.path.isdir(src):
            continue

        os.makedirs(dst, exist_ok=True)

        existing = len(os.listdir(dst))
        needed = TARGET_PER_CLASS - existing

        if needed <= 0:
            continue

        files = [f for f in os.listdir(src) if not f.startswith('.')]
        random.shuffle(files)

        added = 0
        for f in files:
            if added >= needed:
                break

            try:
                shutil.copy(os.path.join(src, f), dst)
                added += 1
            except:
                continue

        print(f"{cls}: +{added}")

In [16]:
# scraping
merge_dataset(extra_path)

# mendeley
merge_dataset(mendeley_path)

Bakso: +92
Capcay: +79
Nasi Uduk: +84
Ayam Geprek: +68
Pecel: +85
Tempe Goreng: +90
gudeg: +400
gado_gado: +383
rendang: +339
bakso: +400
bebek_betutu: +400
nasi_goreng: +400
pempek: +400
sate: +400
rawon: +400
soto: +400


In [17]:
mapping_fix = {
    'bakso': 'Bakso',
    'gado_gado': 'Gado-Gado',
    'nasi_goreng': 'Nasi Goreng',
    'rendang': 'Rendang',
    'rawon': 'Rawon',
    'soto': 'Soto',
    'sate': 'Sate',
    'bebek_betutu': 'Bebek Betutu',
    'pempek': 'Pempek'
}

for old, new in mapping_fix.items():
    src = os.path.join(final_path, old)
    dst = os.path.join(final_path, new)

    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)

        for f in os.listdir(src):
            shutil.move(os.path.join(src, f), dst)

        os.rmdir(src)

        print(f"{old} > {new} merged")

bakso > Bakso merged
gado_gado > Gado-Gado merged
nasi_goreng > Nasi Goreng merged
rendang > Rendang merged
rawon > Rawon merged
soto > Soto merged
sate > Sate merged
bebek_betutu > Bebek Betutu merged
pempek > Pempek merged


#Data Cleaning

In [18]:
valid_ext = ['.jpg', '.jpeg', '.png']

def clean_folder(folder):
    for f in os.listdir(folder):
        path = os.path.join(folder, f)

        # hapus non-image
        if not any(f.lower().endswith(ext) for ext in valid_ext):
            os.remove(path)
            continue

        try:
            img = Image.open(path)
            img.verify()

            # threshold lebih aman
            if img.size[0] < 100 or img.size[1] < 100:
                os.remove(path)

        except:
            os.remove(path)

for cls in os.listdir(final_path):
    clean_folder(os.path.join(final_path, cls))

print("FINAL CLEANED")

FINAL CLEANED


#Distribusi Data

In [19]:
for cls in os.listdir(final_path):
    count = len(os.listdir(os.path.join(final_path, cls)))
    print(cls, count)

Mie Goreng 499
Nasi Goreng 899
gudeg 400
Bakso 492
Burger 500
Ayam Goreng 400
Capcay 79
Rawon 799
Rendang 737
Nasi Uduk 84
French Fries 500
Gado-Gado 781
Donat 500
Ayam Geprek 68
Pecel 85
Bebek Betutu 399
Nasi Padang 400
Pancake 500
Soto 798
Pizza 500
Ikan Goreng 400
Pempek 399
Sate 800
Tempe Goreng 90


#Split Data

In [20]:
source_path = 'data/vision/final'

split_path = 'data/vision/split'

train_path = os.path.join(split_path, 'train')
val_path = os.path.join(split_path, 'val')
test_path = os.path.join(split_path, 'test')

for p in [train_path, val_path, test_path]:
    os.makedirs(p, exist_ok=True)

In [21]:
for cls in os.listdir(source_path):
    cls_path = os.path.join(source_path, cls)

    images = [f for f in os.listdir(cls_path) if not f.startswith('.')]
    random.shuffle(images)

    total = len(images)

    train_split = int(0.8 * total)
    val_split = int(0.1 * total)

    train_files = images[:train_split]
    val_files = images[train_split:train_split + val_split]
    test_files = images[train_split + val_split:]

    # create folder per class
    for base in [train_path, val_path, test_path]:
        os.makedirs(os.path.join(base, cls), exist_ok=True)

    # copy files
    for f in train_files:
        shutil.copy(os.path.join(cls_path, f), os.path.join(train_path, cls, f))

    for f in val_files:
        shutil.copy(os.path.join(cls_path, f), os.path.join(val_path, cls, f))

    for f in test_files:
        shutil.copy(os.path.join(cls_path, f), os.path.join(test_path, cls, f))

    print(f"{cls}: {len(train_files)} train | {len(val_files)} val | {len(test_files)} test")

Mie Goreng: 399 train | 49 val | 51 test
Nasi Goreng: 719 train | 89 val | 91 test
gudeg: 320 train | 40 val | 40 test
Bakso: 393 train | 49 val | 50 test
Burger: 400 train | 50 val | 50 test
Ayam Goreng: 320 train | 40 val | 40 test
Capcay: 63 train | 7 val | 9 test
Rawon: 639 train | 79 val | 81 test
Rendang: 589 train | 73 val | 75 test
Nasi Uduk: 67 train | 8 val | 9 test
French Fries: 400 train | 50 val | 50 test
Gado-Gado: 624 train | 78 val | 79 test
Donat: 400 train | 50 val | 50 test
Ayam Geprek: 54 train | 6 val | 8 test
Pecel: 68 train | 8 val | 9 test
Bebek Betutu: 319 train | 39 val | 41 test
Nasi Padang: 320 train | 40 val | 40 test
Pancake: 400 train | 50 val | 50 test
Soto: 638 train | 79 val | 81 test
Pizza: 400 train | 50 val | 50 test
Ikan Goreng: 320 train | 40 val | 40 test
Pempek: 319 train | 39 val | 41 test
Sate: 640 train | 80 val | 80 test
Tempe Goreng: 72 train | 9 val | 9 test


In [22]:
def count_images(folder):
    total = 0
    for cls in os.listdir(folder):
        total += len(os.listdir(os.path.join(folder, cls)))
    return total

print("Train:", count_images(train_path))
print("Val:", count_images(val_path))
print("Test:", count_images(test_path))

Train: 8883
Val: 1102
Test: 1124


#ZIP Dataset

In [23]:
!zip -r dataset_food_final.zip data/vision/split

Streaming output truncated to the last 5000 lines.
  adding: data/vision/split/train/Donat/133819.jpg (deflated 2%)
  adding: data/vision/split/train/Donat/2627068.jpg (deflated 0%)
  adding: data/vision/split/train/Donat/857351.jpg (deflated 0%)
  adding: data/vision/split/train/Donat/2949511.jpg (deflated 0%)
  adding: data/vision/split/train/Donat/3654140.jpg (deflated 1%)
  adding: data/vision/split/train/Donat/1313900.jpg (deflated 0%)
  adding: data/vision/split/train/Donat/1836327.jpg (deflated 1%)
  adding: data/vision/split/train/Donat/2796001.jpg (deflated 1%)
  adding: data/vision/split/train/Donat/325885.jpg (deflated 12%)
  adding: data/vision/split/train/Donat/1702242.jpg (deflated 0%)
  adding: data/vision/split/train/Donat/2979231.jpg (deflated 0%)
  adding: data/vision/split/train/Donat/2208114.jpg (deflated 1%)
  adding: data/vision/split/train/Donat/2157609.jpg (deflated 1%)
  adding: data/vision/split/train/Donat/2238316.jpg (deflated 1%)
  adding: data/vision/split

#Preprocessing

In [24]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = 'data/vision/split/train'
val_dir = 'data/vision/split/val'
test_dir = 'data/vision/split/test'

In [28]:
!unzip dataset_food_final.zip

Archive:  dataset_food_final.zip
replace data/vision/split/test/Mie Goreng/Mie Goreng_187.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

##Train Resize + Augmentation

In [25]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7,1.3]
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 8883 images belonging to 24 classes.


##Validation Resize

In [26]:
val_datagen = ImageDataGenerator(rescale=1./255)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 1102 images belonging to 24 classes.


##Test Resize

In [27]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 1124 images belonging to 24 classes.
